# Trade Execution

Runs execution strategies against the 5-minute `BINNED_DATA` bins and analyses the results (fill rate, implementation shortfall, cost).

All reusable logic lives in [`execution.py`](execution.py); this notebook is the driver. (`test_snowflake.ipynb` stays the data explorer.)

**Layers** (see `execution.py` docstrings for full detail):

| Function | What it does |
|---|---|
| `simulate_bin_fill` | scalar reference model for a single bin |
| `twap_schedule` | baseline strategy: split quantity evenly across the day's bins |
| `vwap_schedule` | baseline strategy: split quantity by each bin's share of the day's volume |
| `run_strategy` | run one order `(security, date, side, quantity)` → per-bin fills |
| `run_trade_list` | run many orders (the trade list) → per-bin fills with `order_id` |
| `summarise_fills` | collapse to one row per order with fill rate / IS / cost |


In [ ]:
import polars as pl
from execution import (
    twap_schedule,
    vwap_schedule,
    run_strategy,
    run_trade_list,
    summarise_fills,
)

pl.Config.set_tbl_cols(-1)
RAW = "data/raw"

trade_list = pl.read_csv(f"{RAW}/trade_list.csv")
trade_list.head()

## Load bins

For a quick demo we load bins for just a couple of instruments (one liquid, one illiquid). For a full back-test, read the whole table instead — it is ~13M rows / ~2.5 GB in memory:

```python
bins = pl.read_csv(f"{RAW}/binned_data.csv")
```

In [29]:
demo_secs = ["GC2025V Comdty", "PA2022Z Comdty"]  # liquid gold, illiquid palladium

bins = (
    pl.scan_csv(f"{RAW}/binned_data.csv")
    .filter(pl.col("security").is_in(demo_secs))
    .collect()
)
bins

qcode,publication_date,security,bin_start_time,gmt_offset_hours,open,high,low,close,twa_bid_size,twa_ask_size,bid_size_start,bid_size_end,ask_size_start,ask_size_end,volume,signed_volume,trade_count,vwap,twa_bid,twa_ask,bid_start,bid_end,ask_start,ask_end
str,str,str,str,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64
"""GC""","""2025-08-21""","""GC2025V Comdty""","""12:30:00.000000000""",-4.0,3357.0,3357.2,3356.9,3357.1,3.72327,4.416947,7,2,4,1,5,3,5,3357.06,3357.106662,3357.428751,3357.4,3357.0,3357.8,3357.2
"""GC""","""2025-08-21""","""GC2025V Comdty""","""13:05:00.000000000""",-4.0,3358.0,3358.3,3358.0,3358.3,3.769527,4.040703,2,4,5,4,2,-1,2,3358.15,3358.166469,3358.434343,3357.5,3357.9,3357.7,3358.2
"""GC""","""2025-09-26""","""GC2025V Comdty""","""07:15:00.000000000""",-4.0,3743.7,3745.3,3743.3,3745.3,3.585983,3.639973,3,2,5,6,12,7,12,3743.933333,3744.485734,3744.780858,3743.4,3746.2,3743.7,3746.5
"""GC""","""2025-09-05""","""GC2025V Comdty""","""07:30:00.000000000""",-4.0,3576.6,3576.6,3575.6,3575.8,3.80029,3.624047,8,2,2,1,15,2,11,3576.033333,3575.999761,3576.24569,3576.3,3575.8,3576.6,3576.0
"""GC""","""2025-09-05""","""GC2025V Comdty""","""14:10:00.000000000""",-4.0,3621.0,3621.0,3619.2,3619.2,3.153997,3.67401,2,7,2,6,7,2,7,3620.071429,3621.03912,3621.265013,3622.2,3619.0,3622.4,3619.3
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""PA""","""2022-10-20""","""PA2022Z Comdty""","""04:10:00.000000000""",-4.0,1998.0,2001.0,1987.5,1995.0,1.256073,1.59502,1,1,1,2,17,2,15,1996.941176,1993.873782,1998.529693,1991.0,1992.5,1996.0,1997.0
"""PA""","""2022-10-20""","""PA2022Z Comdty""","""04:30:00.000000000""",-4.0,1992.0,1992.0,1992.0,1992.0,1.20146,2.1444,1,1,1,9,1,0,1,1992.0,1989.456387,1993.66039,1987.5,1991.5,1992.0,1996.0
"""PA""","""2022-09-28""","""PA2022Z Comdty""","""15:40:00.000000000""",-4.0,2156.0,2158.0,2155.5,2156.5,1.126773,1.634757,1,1,1,1,6,2,4,2156.25,2154.504405,2157.68185,2154.5,2154.0,2157.5,2158.5


## Single order

Take one real row from the trade list (one bucket for a security-day) and run it. `run_strategy` takes the bins for that `(security, date)` plus the row's `side` and `quantity`, and returns one row per bin.

In [30]:
# pick one real order from the trade list (one row = one bucket for a security-day)
order = trade_list.filter(pl.col("security") == "GC2025V Comdty").row(0, named=True)
print(order)

day = bins.filter(
    (pl.col("security") == order["security"])
    & (pl.col("publication_date") == order["date"])
)

fills = run_strategy(day, side=order["side"], quantity=order["quantity"], strategy=twap_schedule)
fills.head()

{'security': 'GC2025V Comdty', 'date': '2025-07-31', 'trade_list': 'large_buys', 'side': 'buy', 'quantity': 761}


order_id,security,publication_date,qcode,side,order_quantity,bin_start_time,volume,vwap,twa_bid,twa_ask,spread,q_requested,q_filled,q_unfilled,participation_rate,slippage_factor,fill_price,notional,cost,filled,data_available
i32,str,str,str,str,f64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool
0,"""GC2025V Comdty""","""2025-07-31""","""GC""","""buy""",761.0,"""03:00:00.000000000""",23,3323.421739,3322.799493,3323.088507,0.289015,4.847134,4.847134,0.0,0.210745,0.229825,3323.488162,16109.391664,0.32196,true,true
0,"""GC2025V Comdty""","""2025-07-31""","""GC""","""buy""",761.0,"""03:05:00.000000000""",233,3324.727468,3323.683917,3323.962368,0.27845,4.847134,4.847134,0.0,0.020803,0.140789,3324.766671,16115.588766,0.190021,true,true
0,"""GC2025V Comdty""","""2025-07-31""","""GC""","""buy""",761.0,"""03:10:00.000000000""",165,3324.784848,3324.371042,3324.66078,0.289738,4.847134,4.847134,0.0,0.029377,0.148471,3324.827866,16115.885389,0.208512,true,true
0,"""GC2025V Comdty""","""2025-07-31""","""GC""","""buy""",761.0,"""03:15:00.000000000""",19,3323.752632,3323.940178,3324.289575,0.349397,4.847134,4.847134,0.0,0.255112,0.242838,3323.837479,16111.084848,0.411264,true,true
0,"""GC2025V Comdty""","""2025-07-31""","""GC""","""buy""",761.0,"""03:20:00.000000000""",36,3325.483333,3325.593179,3325.900623,0.307444,4.847134,4.847134,0.0,0.134643,0.20377,3325.545981,16119.366189,0.303662,true,true


In [31]:
summarise_fills(fills)

order_id,security,date,qcode,side,order_quantity,n_bins,n_fillable,n_filled,total_volume,total_requested,total_filled,filled_notional,total_cost,arrival_price,terminal_price,fill_rate,avg_fill_price,unfilled_qty,participation_overall,exec_slippage_bps,is_currency,is_bps
i32,str,str,str,str,f64,u32,u32,u32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,"""GC2025V Comdty""","""2025-07-31""","""GC""","""buy""",761.0,157,156,156,7973,761.0,750.764331,2.4970e6,48.263654,3323.421739,3316.984615,0.98655,3325.94194,10.235669,0.094163,7.58315,1826.188311,7.220636


## Run the trade list

The `TRADE_LIST` table has six rows per (security, date) — `small_buys`, `small_sells`, `medium_buys`, `medium_sells`, `large_buys`, `large_sells`. Pass `trade_list=` to run **one bucket at a time** (the usual back-test). Each order gets a unique `order_id` and is auto-tagged with its bucket.

In [32]:
orders = trade_list.filter(pl.col("security").is_in(demo_secs))

# run a single bucket, e.g. medium_buys (one of the six)
all_fills = run_trade_list(bins, orders, trade_list="medium_buys", strategy=twap_schedule)
all_fills.shape

(16929, 23)

In [33]:
summary = summarise_fills(all_fills)
summary.sort("security", "side")

order_id,trade_list,security,date,qcode,side,order_quantity,n_bins,n_fillable,n_filled,total_volume,total_requested,total_filled,filled_notional,total_cost,arrival_price,terminal_price,fill_rate,avg_fill_price,unfilled_qty,participation_overall,exec_slippage_bps,is_currency,is_bps
i32,str,str,str,str,str,f64,u32,u32,u32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
65,"""medium_buys""","""GC2025V Comdty""","""2025-07-31""","""GC""","""buy""",76.0,157,156,156,7973,76.0,75.515924,251156.085013,3.011831,3323.421739,3316.984615,0.993631,3325.869209,0.484076,0.009471,7.364309,181.70692,7.194033
107,"""medium_buys""","""GC2025V Comdty""","""2025-09-29""","""GC""","""buy""",50.0,157,124,124,965,50.0,39.490446,150714.511803,2.322219,3812.15,3821.3,0.789809,3816.48038,10.509554,0.040923,11.359415,267.171038,14.016817
106,"""medium_buys""","""GC2025V Comdty""","""2025-09-26""","""GC""","""buy""",52.0,157,145,145,2439,52.0,48.025478,180544.953201,2.393536,3740.744444,3763.743478,0.923567,3759.357779,3.974522,0.019691,49.758369,985.32445,50.654482
105,"""medium_buys""","""GC2025V Comdty""","""2025-09-25""","""GC""","""buy""",54.0,157,156,156,4274,54.0,53.656051,200831.194049,2.317208,3739.970588,3746.742857,0.993631,3742.936546,0.343949,0.012554,7.930431,161.47091,7.995256
104,"""medium_buys""","""GC2025V Comdty""","""2025-09-24""","""GC""","""buy""",55.0,157,156,156,4140,55.0,54.649682,205235.635349,2.314576,3775.608,3730.85,0.993631,3755.477244,0.350318,0.0132,-53.31792,-1115.818983,-53.73338
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
6,"""medium_buys""","""PA2022Z Comdty""","""2022-09-08""","""PA""","""buy""",16.0,157,127,127,1941,16.0,12.942675,27191.483891,10.053829,2032.806452,2138.277778,0.808917,2100.916816,3.057325,0.006668,335.055827,1203.990431,370.174947
5,"""medium_buys""","""PA2022Z Comdty""","""2022-09-07""","""PA""","""buy""",16.0,157,119,119,1287,16.0,12.127389,24345.456478,8.838248,1986.916667,2033.5,0.757962,2007.477241,3.872611,0.009423,103.479803,429.745226,135.179683
4,"""medium_buys""","""PA2022Z Comdty""","""2022-09-06""","""PA""","""buy""",16.0,157,119,119,1435,16.0,12.127389,24260.940763,8.786417,2056.0,1984.75,0.757962,2000.508246,3.872611,0.008451,-269.901529,-948.893632,-288.452588


### Reading the metrics

- **`fill_rate`** = `total_filled / order_quantity`. Baseline TWAP allocates to every bin including zero-volume ones, so illiquid names (e.g. `PA`) under-fill heavily — the gap a smarter schedule should close.
- **`exec_slippage_bps`** — filled-only slippage vs the arrival price, signed so **positive = worse**.
- **`is_bps`** — implementation shortfall vs arrival price, including opportunity cost on the unfilled remainder marked at the day's terminal price. Captures both spread *and* intraday price drift.

To plug in a new strategy, write a function `my_strategy(bins, quantity, **params) -> Sequence[float]` (per-bin requested quantities) and pass `strategy=my_strategy` to `run_strategy` / `run_trade_list`.

## VWAP strategy

The same evaluation, but with `vwap_schedule` — which splits the order across bins **in proportion to each bin's share of the day's traded volume** (`quantity × volume / total_volume`) instead of evenly. It tracks the intraday volume profile (heavy at the open/close, thin midday) and, crucially, allocates **nothing** to zero-volume bins — so it should fill better than TWAP on the illiquid names where naive TWAP wastes quantity on dead bins.

Everything downstream (fill model, metrics) is identical; only the per-bin allocation changes. Run the same bucket so the two are directly comparable.

In [ ]:
# single order, VWAP — same order as the TWAP single-order example above
vwap_fills = run_strategy(day, side=order["side"], quantity=order["quantity"], strategy=vwap_schedule)
summarise_fills(vwap_fills)

In [ ]:
# run the same bucket through VWAP and summarise
vwap_all_fills = run_trade_list(bins, orders, trade_list="medium_buys", strategy=vwap_schedule)
vwap_summary = summarise_fills(vwap_all_fills)
vwap_summary.sort("security", "side")